In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


# 1．このNotebookの目的
* 目的変数SalePriceの対数変換は、このNotebookでは扱わない。理由は「前処理(X側)の改善効果」だけを純粋に見たいので、yはBaselineと同条件に揃えた方が比較がフェアになるから。

# 2．データ読み込み

In [2]:
#"Id"列をインデックスに指定
import pandas as pd
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv").set_index("Id")
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv").set_index("Id")

# 3. X,yの分離

In [3]:
#目的変数”SalePrice"を取り出す
y = train.SalePrice
X = train.drop(columns=["SalePrice"])

# 4. 前処理対象の列を確認
* EDAの分類をこのNotebookに再掲・再構築
* train/testそれぞれの欠損状況を再確認(testにはtrainで欠損していなかった列にも欠損がある。例: MSZoning, Utilities, BsmtFullBath, GarageCars, KitchenQual, SaleTypeなど)

In [4]:
#4-1 数値列とカテゴリ列に分類
#MSSubClass,MoSoldは数値だが意味的にカテゴリなのでカテゴリ型に変換
numeric_to_category = ["MSSubClass","MoSold"]
X[numeric_to_category] = X[numeric_to_category].astype("category")
test[numeric_to_category] = test[numeric_to_category].astype("category")

num_cols = X.select_dtypes(include="number").columns.to_list()
cat_cols = X.select_dtypes(include=["category","object"]).columns.to_list()


#4-2 カテゴリ列を「順序カテゴリ」「通常カテゴリ」に分類
#edaで確認した、値に大小関係のあるカテゴリ列のこと
ordinal_cat_cols = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual",
    "Functional", "FireplaceQu", "GarageFinish", "GarageQual",
    "GarageCond", "PavedDrive", "PoolQC"
]

normal_cat_cols = [col for col in cat_cols
                  if col not in ordinal_cat_cols]


#4-3 欠損の意味による分類(absence,unknown,drop候補)

#欠損が「設備無し」を表す列→None埋め
#　※ordinal_cat_colsとnormal_cat_colsの両方にまたがる点に注意
absence_cat_cols = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature"
]

#本当に値が不明な列→最頻値埋め候補
unknown_cat_cols = [col for col in cat_cols 
                   if X[col].isna().any()
                   and col not in absence_cat_cols]

#欠損率が非常に高く、情報量も少ない列→削除かNone埋め候補
drop_candidate = ["Alley", "PoolQC", "Fence", "MiscFeature"]


#4-4 train/test両方の欠損数を比較する
all_cols = num_cols + cat_cols

train_missing = X[all_cols].isna().sum()
test_missing = test[all_cols].isna().sum()

missing_compare = pd.DataFrame({
    "missing_train":train_missing,
    "missing_test":test_missing
})

#trainでは欠損0だが、testでは欠損がある列だけ抽出
test_only_missing = missing_compare[
    (missing_compare["missing_train"] == 0) &
    (missing_compare["missing_test"]>0)
]

test_only_missing

,missing_train,missing_test
BsmtFinSF1,0,1
BsmtFinSF2,0,1
BsmtUnfSF,0,1
TotalBsmtSF,0,1
BsmtFullBath,0,2
BsmtHalfBath,0,2
GarageCars,0,1
GarageArea,0,1
MSZoning,0,4
Utilities,0,2


## trainには無いが、testデータには欠損がある列
上記の特徴量は、trainデータで学習させただけでは、testデータでの欠損埋めがうまくいかない。上記の特徴量も、testでは欠損扱いなので、欠損の種類によって分類する必要がある。

In [5]:
#4-5:testだけ欠損している列を、数値/カテゴリに仕分ける
test_only_num_cols = [col for col in test_only_missing.index if col in num_cols]
test_only_cat_cols = [col for col in test_only_missing.index if col in cat_cols]

#4-6:testのみ欠損のカテゴリ列を、unknown_cat_colsに追加する
unknown_cat_cols = list(dict.fromkeys(
    unknown_cat_cols +
    [col for col in test_only_cat_cols if col not in absence_cat_cols]
))

unknown_cat_cols

['MasVnrType',
 'Electrical',
 'MSZoning',
 'Utilities',
 'Exterior1st',
 'Exterior2nd',
 'KitchenQual',
 'Functional',
 'SaleType']

# 5. 前処理方針を決定(eda参照)
## 数値列:
* LotFrontage:平均値埋め
* 歪度の高い(0.75以上)連続数値列(skewed_cols):log1p変換
* 数値列も”欠損の意味”によって分類
  >absence_num_cols:”0”埋め
  >
  >unknown_num_cols:”中央値”埋め
## カテゴリ列:
* 順序カテゴリ列:欠損”None”埋め
  >OrdinalEncoderを使い、各列ごとに「Excellent > Good > Average > Fair > Poor」等の順序配列を手動定義。欠損は"None"として最下位カテゴリに含める
* 通常カテゴリ列:
  >absence_cat_cols:"None"埋め→OneHot
  >
  >unknown_cat_cols:最頻値埋め→OneHot
* drop_candidate: 完全に削除するより、情報量ゼロではないので"None"埋め+OneHot(または順序カテゴリとして残す)。削除したものと比較

In [6]:
#5-1 数値列の方針 EDAで算出済みのリストを再利用
skewed_cols = ['MiscVal', 'LotArea', '3SsnPorch', 'LowQualFinSF',
               'BsmtFinSF2', 'ScreenPorch', 'EnclosedPorch', 'MasVnrArea', 
               'OpenPorchSF', 'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF', 
               '1stFlrSF', 'GrLivArea', 'BsmtUnfSF', '2ndFlrSF']
lotfrontage_policy = "impute_mean"  
log_transform_cols = skewed_cols  

#5-2 構造的に0が正しい列(地下室・ガレージ関連)→欠損:0埋め方針
absence_num_cols = [
    "BsmtFullBath", "BsmtHalfBath",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "GarageCars", "GarageArea"
]

#本当に値が不明な数値列→欠損:中央値埋め方針(外れ値を考慮しmeanではなくmedian採用)
unknown_num_cols = [
    col for col in num_cols
    if (train_missing.get(col, 0)>0 or test_missing.get(col, 0)>0)
    and col not in absence_num_cols
]

#5-3 順序カテゴリ列の方針
#各順序カテゴリ列ごとに、"Poor"→"Excellent”などの順序配列を手動変換
# ①absence型：欠損=設備なし → "None"を最下位に含める順序配列
# (Poor → Excellentの5段階評価が基本形)
quality_scale = ["None", "Po", "Fa", "TA", "Gd", "Ex"]

ordinal_orders = {
    # --- ①absence型(欠損="None"、Noneを最下位に含む) ---
    "BsmtQual":     quality_scale,                              # 地下室の仕上がり品質
    "BsmtCond":     quality_scale,                              # 地下室の状態
    "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],           # 地下室の外への露出度
    "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],  # 地下室の主要仕上げ
    "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],  # 地下室の追加仕上げ
    "FireplaceQu":  quality_scale,                              # 暖炉の品質
    "GarageFinish": ["None", "Unf", "RFn", "Fin"],              # ガレージの内装仕上げ
    "GarageQual":   quality_scale,                              # ガレージの品質
    "GarageCond":   quality_scale,                              # ガレージの状態
    "PoolQC":       ["None", "Fa", "TA", "Gd", "Ex"],           # プールの品質(Poレベルは存在しない)

    # --- ②unknown型(欠損=最頻値で埋める、Noneは含めない) ---
    "ExterQual":  ["Po", "Fa", "TA", "Gd", "Ex"],   # 外壁材の品質
    "ExterCond":  ["Po", "Fa", "TA", "Gd", "Ex"],   # 外壁材の状態
    "HeatingQC":  ["Po", "Fa", "TA", "Gd", "Ex"],   # 暖房設備の品質
    "KitchenQual":["Po", "Fa", "TA", "Gd", "Ex"],   # キッチンの品質
    "PavedDrive": ["N", "P", "Y"],                  # 車道の舗装状況(Dirt→Partial→Paved)
    "Functional": ["Sal","Sev","Maj2","Maj1","Mod","Min2","Min1","Typ"], # 家の機能性(Typ=標準が最高評価)
}


#5-4 通常カテゴリ列の方針
#absence_cat_cols → "None"埋め → OneHot
#unknown_cat_cols → 最頻値埋め → OneHot

#5-5 drop_candidateの方針
#完全削除より、None埋め+OneHotを基本

#5-6 最終的な処理方針テーブル
def get_treatment(col):
    if col in absence_cat_cols:
        return "fill_none"
    elif col in absence_num_cols:
        return "fill_zero"
    elif col in unknown_cat_cols:
        return "fill_most_frequent"
    elif col in unknown_num_cols:
        return "fill_median"
    elif col in drop_candidate:
        return "fill_none_onehot"
    elif col == "LotFrontage":
        return "impute_mean"
    else:
        return "no_missing"

preprocessing_plan = pd.DataFrame({"col":cat_cols + num_cols})
preprocessing_plan["treatment"] = preprocessing_plan["col"].apply(get_treatment)
preprocessing_plan["log_transform"] = preprocessing_plan["col"].isin(log_transform_cols)
preprocessing_plan

,col,treatment,log_transform
0,MSSubClass,no_missing,False
1,MSZoning,fill_most_frequent,False
2,Street,no_missing,False
3,Alley,fill_none,False
4,LotShape,no_missing,False
...,...,...,...
74,3SsnPorch,no_missing,True
75,ScreenPorch,no_missing,True
76,PoolArea,no_missing,False
77,MiscVal,no_missing,True


# 6. 数値列パイプライン実装
* 数値列は「欠損の埋め方が違う2グループ」をColumnTransformerで分けて処理し、その後まとめてlog1p変換をかける、という2段構えのパイプラインにする
* log1p(単なるlogではない)を使う理由は、0を含む列でもエラーにならないため
* RandomForestは「木構造」で分岐するモデルなので、標準化(スケーリング)は不要。スケールは単位の大きさのこと
  >決定事項 対象列
  >
  >①0埋め absence_num_cols(地下室・ガレージ関連)
  >
  >②中央値埋め	unknown_num_cols + LotFrontage
  >
  >③log1p変換	log_transform_cols(EDAのskewed_cols)
  >
  >④標準化	なし(RandomForestのため)

# 7. カテゴリ列パイプライン実装
* 順序カテゴリ用パイプライン(欠損→"None"、OrdinalEncoder(categories=[...手動定義...]))
* 通常カテゴリ用パイプライン(absence/unknownで補完方法を分けるなら、さらにColumnTransformerをネストするか、fillna処理を先に一括で済ませてからOneHotにまとめるかを決める)

# 8. ColumnTransformer
* 数値/順序カテゴリ/通常カテゴリの3系統をまとめる(Baselineは数値/カテゴリの2系統だったので、ここが明確な進化ポイント)

# 9. Pipeline
* モデルはBaselineと全く同じ設定(RandomForestRegressor(n_jobs=-1, random_state=10, n_estimators=300))を使う。これにより「前処理を変えたことだけ」がスコア変化の要因になり、比較の妥当性が保てる

# 10. 前処理後データの確認
* preprocessor.fit_transform(X)の出力shape確認、欠損が無いこと確認、get_feature_names_out()で列名確認
* 対数変換前後の歪度比較(EDAのskewnessと比較)
* 注意点として明記: ここでの確認はあくまで可視化目的で、実際のCVではPipelineごとcross_val_scoreに渡すことでfold内で正しくfit(リーク防止)されている、という点をひとこと書いておくと良い

# 11. Baselineとのスコア比較
* CV設定(KFold(n_splits=5, shuffle=True, random_state=10), scoring="neg_root_mean_squared_log_error")をBaselineと完全に同一にする
* Baseline: 0.1466 → 今回の結果を並べて表かグラフで比較
* 改善/悪化した場合、それぞれの前処理変更(順序エンコード化、log変換、欠損処理変更)のうちどれが効いたのか、可能であれば要素ごとにアブレーション(1つずつ変更して比較)すると根拠が強くなる

# 12. まとめ
* Baseline比でのCVスコア変化と、次のFeature Engineering Notebookへの申し送り事項(例: 目的変数の対数変換をここで試す、など)